# Ch.5 — Matrices, Linear Systems, and Matrix Calculus

**Running theme.** From one $(t, y)$ sample to 500. From one slider-fit parabola to one `lstsq` call recovering the laws of motion. Plus eight shot features predicting makes vs misses.

**Learning outcomes.**
1. See the same $A\mathbf{x}$ computed three ways — row, column, transformation — and confirm they agree.
2. Watch a $2 \times 2$ matrix warp the unit square via a slider, in real time.
3. Fit the full free-throw trajectory using the normal equations, and recover $h_0$, $v_{0y}$, $-g/2$.
4. Build a multi-feature shot predictor with 8 inputs and compare `solve`, `lstsq`, and `inv`-based approaches.

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch
from ipywidgets import FloatSlider, FloatText, HBox, VBox, Output, jslink

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (7.5, 5.0), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})
np.set_printoptions(precision=3, suppress=True)

def linked_pair(label, value, vmin, vmax, step=0.05):
    sl = FloatSlider(value=value, min=vmin, max=vmax, step=step,
                     description=label, continuous_update=True, readout=False)
    tx = FloatText(value=value, description=" ", step=step,
                   layout={"width": "110px"})
    jslink((sl, "value"), (tx, "value"))
    return sl, tx

## 2 · Three views of $A\mathbf{x}$ — all the same number

Row: dot products into a vector. Column: weighted sum of columns. Transformation: $A$ warps space and $A\mathbf{x}$ is the image of $\mathbf{x}$.

In [ ]:
A = np.array([[1.5, 0.5],
              [0.3, 1.2]])
x = np.array([0.7, 0.4])

# 1. @ operator (preferred, unambiguous)
b1 = A @ x

# 2. Row view: dot each row of A with x
b2 = np.array([A[0] @ x, A[1] @ x])

# 3. Column view: weighted sum of columns of A
b3 = x[0] * A[:, 0] + x[1] * A[:, 1]

print(f"A @ x                 = {b1}")
print(f"row view              = {b2}")
print(f"column view (sum)     = {b3}")
print(f"all three agree?      = {np.allclose(b1, b2) and np.allclose(b1, b3)}")

## 3 · Interactive — drag a $2 \times 2$ matrix and watch the unit square deform

Each entry of $A$ is a slider. The blue square is fixed. The orange parallelogram is its image under $A$; its area equals $|\det(A)|$. Flip the sign of any diagonal entry and watch the orientation reverse.

In [ ]:
a11_sl, a11_tx = linked_pair("A[0,0]", value=1.5, vmin=-2.0, vmax=2.5, step=0.05)
a12_sl, a12_tx = linked_pair("A[0,1]", value=0.5, vmin=-2.0, vmax=2.0, step=0.05)
a21_sl, a21_tx = linked_pair("A[1,0]", value=0.3, vmin=-2.0, vmax=2.0, step=0.05)
a22_sl, a22_tx = linked_pair("A[1,1]", value=1.2, vmin=-2.0, vmax=2.5, step=0.05)
det_out = Output()

square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]])

fig, ax = plt.subplots(figsize=(6.0, 6.0))
orig_patch = Polygon(square[:-1], closed=True, facecolor="#D6EAF8",
                     edgecolor="#2E86C1", lw=2.0, alpha=0.9)
img_patch  = Polygon(square[:-1], closed=True, facecolor="#FDEBD0",
                     edgecolor="#E67E22", lw=2.0, alpha=0.7)
ax.add_patch(orig_patch); ax.add_patch(img_patch)
ax.axhline(0, color="#7F8C8D", lw=0.5); ax.axvline(0, color="#7F8C8D", lw=0.5)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")

def redraw_warp(*_):
    A_cur = np.array([[a11_sl.value, a12_sl.value],
                      [a21_sl.value, a22_sl.value]])
    img = square @ A_cur.T
    img_patch.set_xy(img[:-1])
    det = float(np.linalg.det(A_cur))
    with det_out:
        det_out.clear_output(wait=True)
        print(f"A =\n{A_cur}")
        print(f"det(A) = {det:+.4f}")
        if abs(det) < 1e-3:
            print("  → A is (nearly) SINGULAR — squashes to a line.")
        elif det < 0:
            print("  → det < 0: orientation is FLIPPED.")
    fig.canvas.draw_idle()

for w in (a11_sl, a12_sl, a21_sl, a22_sl):
    w.observe(redraw_warp, names="value")
redraw_warp()

display(VBox([HBox([a11_sl, a11_tx, a12_sl, a12_tx]),
              HBox([a21_sl, a21_tx, a22_sl, a22_tx]),
              det_out]))

## 4 · Fit the free-throw trajectory with the normal equations

Build the design matrix $X$ with columns $[\mathbf{1}, \mathbf{t}, \mathbf{t}^2]$, solve $\hat{\mathbf{w}} = (X^\top X)^{-1} X^\top \mathbf{y}$, read off physics.

In [ ]:
rng = np.random.default_rng(0)
v0y, h0, g = 7.2, 2.10, 9.81

t = np.linspace(0.05, 1.40, 25)
y_true = h0 + v0y * t - 0.5 * g * t ** 2
y_noisy = y_true + rng.normal(0, 0.05, size=t.shape)

# Design matrix
X = np.column_stack([np.ones_like(t), t, t ** 2])
print(f"X.shape = {X.shape}   (N × d)")
print(f"first 3 rows of X:\n{X[:3]}")

In [ ]:
# Method 1: np.linalg.lstsq — the textbook-recommended way.
w_lstsq, *_ = np.linalg.lstsq(X, y_noisy, rcond=None)

# Method 2: np.linalg.solve on the normal equations — equivalent, faster path.
w_solve = np.linalg.solve(X.T @ X, X.T @ y_noisy)

# Method 3: explicit inverse — DON'T do this in practice. Shown for comparison.
w_inv = np.linalg.inv(X.T @ X) @ X.T @ y_noisy

print(f"lstsq  : ŵ = {w_lstsq}")
print(f"solve  : ŵ = {w_solve}")
print(f"inv    : ŵ = {w_inv}")
print(f"physics: [h0={h0}, v0y={v0y}, -g/2={-g/2:.3f}]")
print(f"\nmax |lstsq - solve| = {np.max(np.abs(w_lstsq - w_solve)):.2e}")
print(f"max |lstsq - inv|   = {np.max(np.abs(w_lstsq - w_inv)):.2e}")

In [ ]:
t_plot = np.linspace(0, 1.55, 300)
y_fit = w_lstsq[0] + w_lstsq[1] * t_plot + w_lstsq[2] * t_plot ** 2
y_truth_plot = h0 + v0y * t_plot - 0.5 * g * t_plot ** 2

fig2, ax2 = plt.subplots()
ax2.axhline(3.05, color="#27AE60", lw=1.0, linestyle=":", label="hoop rim")
ax2.plot(t_plot, y_truth_plot, color="#7F8C8D", lw=1.0, alpha=0.7, label="true physics")
ax2.plot(t_plot, y_fit, color="#8E44AD", lw=2.2, label="normal-equations fit")
ax2.scatter(t, y_noisy, s=32, color="#2C3E50", zorder=5, label="noisy samples")
ax2.set_xlabel("t (s)"); ax2.set_ylabel("y (m)"); ax2.legend()
ax2.set_title("One matrix solve, physics constants back out")
plt.show()

## 5 · Why you should never use `np.linalg.inv` in practice

Construct a poorly-conditioned matrix (high-degree polynomial features on raw $t$) and watch `inv` and `solve` disagree at scale.

In [ ]:
for degree in [2, 4, 6, 8, 10]:
    X_bad = np.column_stack([t ** k for k in range(degree + 1)])
    cond = np.linalg.cond(X_bad.T @ X_bad)
    w_a = np.linalg.lstsq(X_bad, y_noisy, rcond=None)[0]
    try:
        w_b = np.linalg.inv(X_bad.T @ X_bad) @ X_bad.T @ y_noisy
        disagree = np.max(np.abs(w_a - w_b))
    except np.linalg.LinAlgError:
        disagree = float("inf")
    print(f"degree {degree:>2}   cond(XᵀX) = {cond:.2e}   max|lstsq - inv| = {disagree:.2e}")

print("\nLesson: condition number explodes with polynomial degree.")
print("Once it exceeds ~10¹², inv starts giving nonsense; lstsq stays stable.")

## 6 · Multi-feature regression — eight shot inputs, one prediction

Simulate 500 free throws with eight measured features. The made-or-missed *landing height* at the hoop is a linear function of the features (plus Gaussian noise). Fit and read off the coefficients.

In [ ]:
features = ["release speed", "release angle", "release height",
            "wind x", "wind y", "spin rate", "court altitude", "fatigue"]

N, d = 500, 8
true_w = np.array([0.62, 0.09, 0.45, -0.30, 0.05, 0.02, -0.01, -0.12])
true_b = -2.10

X_multi = rng.normal(size=(N, d)) * np.array([1.0, 8.0, 0.3, 1.2, 0.5, 100, 300, 1.5])
X_multi += np.array([7.5, 52.0, 2.10, 0.0, 0.0, 200, 500, 3.0])     # offsets

y_multi = X_multi @ true_w + true_b + rng.normal(0, 0.15, size=N)

# Augment X with a bias column so we estimate b inline with w.
X_aug = np.column_stack([X_multi, np.ones(N)])
w_aug, *_ = np.linalg.lstsq(X_aug, y_multi, rcond=None)
w_est, b_est = w_aug[:-1], w_aug[-1]

print(f"{'feature':<20}  true      fitted   |err|")
print("-" * 50)
for name, wt, we in zip(features, true_w, w_est):
    print(f"{name:<20}  {wt:+.4f}   {we:+.4f}   {abs(we - wt):.4f}")
print(f"{'(bias)':<20}  {true_b:+.4f}   {b_est:+.4f}   {abs(b_est - true_b):.4f}")

# In-sample residual: how close do predictions get?
y_pred = X_aug @ w_aug
rmse = np.sqrt(np.mean((y_pred - y_multi) ** 2))
print(f"\nin-sample RMSE: {rmse:.4f}   (noise σ = 0.15)")

## 7 · Matrix calculus — verify the gradient identities numerically

Check that $\nabla_\mathbf{w} \|X\mathbf{w} - \mathbf{y}\|^2 = 2 X^\top(X\mathbf{w} - \mathbf{y})$ using finite differences. If the analytical expression is right, the finite-difference estimate should match to ~6 digits.

In [ ]:
# Take a random w and compute both sides.
w_test = rng.normal(size=d + 1)

def loss(w):
    return float(np.sum((X_aug @ w - y_multi) ** 2))

analytic_grad = 2 * X_aug.T @ (X_aug @ w_test - y_multi)

h = 1e-5
numeric_grad = np.empty_like(w_test)
for j in range(len(w_test)):
    w_plus, w_minus = w_test.copy(), w_test.copy()
    w_plus[j]  += h
    w_minus[j] -= h
    numeric_grad[j] = (loss(w_plus) - loss(w_minus)) / (2 * h)

rel_err = np.linalg.norm(analytic_grad - numeric_grad) / np.linalg.norm(analytic_grad)
print(f"analytic[:4]  : {analytic_grad[:4]}")
print(f"numeric [:4]  : {numeric_grad[:4]}")
print(f"relative error: {rel_err:.2e}    (should be ≪ 1)")

## 8 · Where this reappears

- **Pre-Req Ch.6.** The gradient expression $2 X^\top(X\mathbf{w} - \mathbf{y})$ was derived via the matrix chain rule you're about to meet formally.
- **ML Ch.1 Linear Regression.** Replace `np.linalg.lstsq` with gradient descent and you have the full ML chapter.
- **ML Ch.4 Neural Networks.** Every layer is $\mathbf{h}_{k+1} = \sigma(W_k \mathbf{h}_k + \mathbf{b}_k)$. Training just does the normal-equations dance repeatedly, with a non-linearity.
- **ML Ch.13 Dimensionality Reduction.** SVD of $X$ gives PCA. Same matrix, different decomposition.
- **ML Ch.18 Transformers.** Attention scores are $Q K^\top$ — a matrix-matrix product you're now equipped to read.

**Final exercise.** Add an L2-regularisation term $\lambda \|\mathbf{w}\|^2$ to the loss and re-derive the closed form: $\hat{\mathbf{w}} = (X^\top X + \lambda I)^{-1} X^\top \mathbf{y}$. Implement it and plot the fitted coefficients as $\lambda$ sweeps from $10^{-4}$ to $10^{4}$. This is **ridge regression** — ML Ch.6 Regularisation.